# executorlib
## Python Standard Library
executorlib is based on the `Executor` interface defined in the [Python Standard Library](https://docs.python.org/3/library/concurrent.futures.html). The Python Standard Library, defines two interfaces:
* `ThreadPoolExecutor` for executing individual Python functions in separate threads.
* `ProcessPoolExecutor` for executing individual Python functions in separate processes.

Let's take the following example:

In [1]:
result_lst = []
for i in range(1, 5):
    result_lst.append(sum([i, i]))

result_lst

[2, 4, 6, 8]

This can be rewritten using either the `ThreadPoolExecutor` or the `ProcessPoolExecutor`:

In [2]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[True, True, True, False]
[2, 4, 6, 8]


In [3]:
from concurrent.futures import ProcessPoolExecutor

with ProcessPoolExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


By using the [concurrent.futures.Future](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Future) objects we can develop asynchronous Python scripts. Specifically the Python functions are executed in the background until `result()` is called. 

## Executor Types
Executorlib currently implements five different Executor classes: 

| Name | Communication | Application | Internal Command |
|------|---------------|-------------|------------------|
| `SingleNodeExecutor` | Socket | Workstation | `Popen()` |
| `SlurmClusterExecutor` | Files | HPC | `sbatch` |
| `SlurmJobExecutor` | Socket | HPC | `srun` |
| `FluxClusterExecutor` | Files | HPC | `flux submit` |
| `FluxJobExecutor` | Socket | HPC | `flux run` |

In the following the `SingleNodeExecutor`, `FluxClusterExecutor` and `FluxJobExecutor` are introduced. The `SlurmClusterExecutor` behave analog to the `FluxClusterExecutor` and the `SlurmJobExecutor` behaves analog to the `FluxJobExecutor`, so they are not introduced here as SLURM is not available in the demonstration environment. 

## SingleNodeExecutor
The `SingleNodeExecutor` is designed to start developing HPC workflows on your workstation. It internally works similar to the `ProcessPoolExecutor` but supports a number of the more advanced functions introduced below. To switch from the `ProcessPoolExecutor` to the `SingleNodeExecutor` you just have to update the name of the executor.

In [4]:
from executorlib import SingleNodeExecutor

with SingleNodeExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


## FluxClusterExecutor
The `FluxClusterExecutor` submits Python functions from the login node to individual compute nodes. It is using file based communication, so the executor can be stopped while the functions are still waiting in the queue.

In [5]:
from executorlib import FluxClusterExecutor

with FluxClusterExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


In [6]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒ4iy2NzT jovyan   executorl+ CD      1      1   1.254s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4fhFxvK jovyan   executorl+ CD      1      1   1.162s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4cYuVFu jovyan   executorl+ CD      1      1   1.151s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4ZP52K9 jovyan   executorl+ CD      1      1   1.162s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0


### Cache
The internal cache of the `FluxClusterExecutor` can be defined using the `cache_directory` parameter. To analyse the content of the current cache directory, we can use the `get_cache_data()` function:

In [7]:
import pandas
from executorlib import get_cache_data

pandas.DataFrame(get_cache_data(cache_directory="executorlib_cache"))

,function,input_args,input_kwargs,output,resource_dict,runtime,queue_id,error_log_file,filename
0,<built-in function sum>,"[[1, 1]]",{},2,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001279,135459241984,None,/home/jovyan/executorlib_cache/sum5684c948e337...
1,<built-in function sum>,"[[3, 3]]",{},6,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001323,139603214336,None,/home/jovyan/executorlib_cache/sumc17e02a6dc40...
2,<built-in function sum>,"[[4, 4]]",{},8,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001545,141750697984,None,/home/jovyan/executorlib_cache/sumf3976430477e...
3,<built-in function sum>,"[[2, 2]]",{},4,"{'cores': 1, 'threads_per_core': 1, 'gpus_per_...",0.001467,137539616768,None,/home/jovyan/executorlib_cache/sum3d89b9dcf89b...


### Submission Template 
While for flux it is commonly not necessary to specify specific job scheduler configurations, this is usually the case with the `SlurmClusterExecutor` as SLURM is configured differently on every HPC. Internally executorlib uses the Python Simple Queuing System [pysqa](https://pysqa.readthedocs.io/) interface: 

In [8]:
submission_template = """\
#!/bin/bash
# flux: --job-name={{job_name}}
# flux: --env=CORES={{cores}}
# flux: --output=time.out
# flux: --error=error.out
# flux: --nslots={{cores}}
{%- if run_time_max %}
# flux: --time-limit={{run_time_max}}s
{%- endif %}
{%- if dependency_list %}
{%- for jobid in dependency_list %}
# flux: --dependency=afterok:{{jobid}}
{%- endfor %}
{%- endif %}

{{command}}
"""

In [9]:
from executorlib import FluxClusterExecutor

with FluxClusterExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i], resource_dict={"submission_template": submission_template}))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


When the same function is submitted with the same parameters it is not computed again, even if the `resource_dictionary` is updated. 

### MPI parallel functions
Besides the option to specify the submission script the `resource_dictionary` also defines the computing resources. Here I demonstrate this for the number of CPU cores to use with an MPI parallel Python function. Still the same also applies for the number of threads of GPUs, basically all the resources which can be specified in the job scheduler can be assigned to individual Python functions using the `resource_dictionary`. A detailed list of the parameters for the `resource_dictionary` is available in the [documentation](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary).

In [10]:
def calc_mpi(i):
    from mpi4py import MPI

    size = MPI.COMM_WORLD.Get_size()
    rank = MPI.COMM_WORLD.Get_rank()
    return i, size, rank

In [11]:
from executorlib import FluxClusterExecutor

with FluxClusterExecutor(pmi_mode="pmix") as exe:
    fs = exe.submit(calc_mpi, 3, resource_dict={"cores": 2})
    print(fs.result())

[(3, 2, 0), (3, 2, 1)]


The advantage of defining the computing resources on a per Python function level is that it is much easier to parallize your code as you only have to modify the functions which actually benefit from the parallel execution, while the rest of the code remains serial.

## FluxJobExecutor
In contrast to the `FluxClusterExecutor` the `FluxJobExecutor` and the `SlurmJobExecutor` both communicate via Sockets, which provides a performance benefit:

In [12]:
from executorlib import FluxJobExecutor

with FluxJobExecutor() as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


In [13]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒCGPpUZW jovyan   python     CD      1      1   0.167s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒCGPpUZV jovyan   python     CD      1      1   0.162s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒCDnbjpK jovyan   python     CD      1      1   0.199s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒCDp5j6f jovyan   python     CD      1      1   0.158s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒAA1PmGT jovyan   executorl+ CD      1      1   2.661s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4iy2NzT jovyan   executorl+ CD      1      1   1.254s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4fhFxvK jovyan   executorl+ CD      1      1   1.162s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4cYuVFu jovyan   executorl+ CD      1      1   1.151s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4ZP52K9 jovyan   executorl+ CD      1      1   1.162s jupyter-janj-lanl-exe

### Block Allocation
To further improve the performance of the evaluation of the Python functions, we can activate `block_allocation=True`. In this mode a single Python process is reused to evaluate multiple Python functions, breaking the previous mapping of one Python function per Python process.

In [14]:
from executorlib import FluxJobExecutor

with FluxJobExecutor(block_allocation=True, max_workers=1) as exe:
    future_lst = []
    for i in range(1, 5):
        future_lst.append(exe.submit(sum, [i, i]))

    print([f.done() for f in future_lst])

    print([f.result() for f in future_lst])

[False, False, False, False]
[2, 4, 6, 8]


In [15]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒELUpJVV jovyan   python     CD      1      1   0.175s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒCGPpUZW jovyan   python     CD      1      1   0.167s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒCGPpUZV jovyan   python     CD      1      1   0.162s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒCDnbjpK jovyan   python     CD      1      1   0.199s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒCDp5j6f jovyan   python     CD      1      1   0.158s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒAA1PmGT jovyan   executorl+ CD      1      1   2.661s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4iy2NzT jovyan   executorl+ CD      1      1   1.254s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4fhFxvK jovyan   executorl+ CD      1      1   1.162s jupyter-janj-lanl-executorlib-tutorial-l9gs95k0
    ƒ4cYuVFu jovyan   executorl+ CD      1      1   1.151s jupyter-janj-lanl-exe

This functionality can be very helpful when a large number of similar computations are executed one after another. 

In [16]:
import pandas
from executorlib import FluxJobExecutor

df = pandas.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})

def sum_df(a, b):
    return a + b

with FluxJobExecutor(block_allocation=True, max_workers=1) as exe:
    df["c"] = list(exe.map(sum_df, df["a"], df["b"]))
    print(df)

   a  b  c
0  1  4  5
1  2  5  7
2  3  6  9


### Initialization function
Beyond just reusing the same Python process for multiple Python functions, we can also pre-load a dataset in this Python process and then access this pre-loaded dataset in multiple Python functions. 

In [17]:
with open("large.txt", "w") as f:
    for i in range(100):
        f.writelines(str(i) + "\n")

In [18]:
def init_file_handle():
    def open_large_file():
        with open("large.txt", "r") as f:
            for line in f:
                yield line.rstrip()
    return {"file_handle": open_large_file()}

In [19]:
def read_next_n_lines(file_handle, n=5):
    return [next(file_handle) for i in range(n)]  

In [20]:
with FluxJobExecutor(block_allocation=True, max_workers=1, init_function=init_file_handle) as exe:
    f1 = exe.submit(read_next_n_lines)
    f2 = exe.submit(read_next_n_lines, n=10)
    print(f1.result(), f2.result())

['0', '1', '2', '3', '4'] ['5', '6', '7', '8', '9', '10', '11', '12', '13', '14']
